# CrewAI Demo

CrewAI orchestrates role-based multi-agent teams that mimic real-world human crew dynamics.

## Required Environment Variables

```
OPENAI_API_KEY
SERPER_API_KEY  # Optional, for web search
```

In [ ]:
# If running standalone without the project setup:
# pip install crewai crewai-tools openai python-dotenv

In [1]:
import os
import yaml
from pathlib import Path
from dotenv import load_dotenv
from typing import Any, Type, Optional
from pydantic import BaseModel, Field

# Load environment variables
load_dotenv(override=True)

# Import CrewAI components
from crewai import Agent, Task, Crew, Process, LLM
from crewai_tools import SerperDevTool
from crewai.tools import BaseTool

d:\Dev\archaas\agentic-ai-course\3-module\.venv\Lib\site-packages\pydantic\_internal\_config.py:323: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  warnings.warn(DEPRECATION_MESSAGE, DeprecationWarning)


## What We're Building

A **Content Marketing Team** — three agents (researcher, writer, editor) collaborating to produce a blog post about AI-powered content marketing.

In [2]:
llm = LLM(model="gpt-4o", temperature=0.7)

web_search_tool = None
try:
    if os.getenv('SERPER_API_KEY'):
        web_search_tool = SerperDevTool()
        print("Web Search Tool (SerperDev) initialized")
    else:
        print("No SERPER_API_KEY found — agents will use LLM knowledge only")
except Exception as e:
    print(f"Web Search Tool error: {e}")

No SERPER_API_KEY found — agents will use LLM knowledge only


## Agent Definition

Agents are role-based team members with a **role**, **goal**, **backstory**, and **tools**.

In [3]:
agents_yaml_content = """
content_researcher:
  role: Senior Content Researcher
  goal: Research and gather comprehensive information on given topics to support content creation
  backstory: Experienced researcher with 10+ years in digital marketing, skilled at finding trends and actionable insights.
  tools:
  - web_search
  verbose: true
  allow_delegation: false

content_writer:
  role: Senior Content Writer
  goal: Create engaging, high-quality content based on research and brand guidelines
  backstory: Seasoned content writer who transforms research into compelling marketing narratives.
  tools: []
  verbose: true
  allow_delegation: false

content_editor:
  role: Senior Content Editor
  goal: Review, refine, and ensure content quality meets brand standards and objectives
  backstory: Experienced editor with a keen eye for detail, brand consistency, and audience engagement.
  tools: []
  verbose: true
  allow_delegation: true
"""

agents_config = yaml.safe_load(agents_yaml_content.strip())

agents_config

{'content_researcher': {'role': 'Senior Content Researcher',
  'goal': 'Research and gather comprehensive information on given topics to support content creation',
  'backstory': 'Experienced researcher with 10+ years in digital marketing, skilled at finding trends and actionable insights.',
  'tools': ['web_search'],
  'verbose': True,
  'allow_delegation': False},
 'content_writer': {'role': 'Senior Content Writer',
  'goal': 'Create engaging, high-quality content based on research and brand guidelines',
  'backstory': 'Seasoned content writer who transforms research into compelling marketing narratives.',
  'tools': [],
  'verbose': True,
  'allow_delegation': False},
 'content_editor': {'role': 'Senior Content Editor',
  'goal': 'Review, refine, and ensure content quality meets brand standards and objectives',
  'backstory': 'Experienced editor with a keen eye for detail, brand consistency, and audience engagement.',
  'tools': [],
  'verbose': True,
  'allow_delegation': True}}

In [4]:
tools_map = {'web_search': web_search_tool}

agents = {}
for agent_name, config in agents_config.items():
    agent_tools = [tools_map[t] for t in config.get('tools', []) if t in tools_map and tools_map[t]]

    backstory = config['backstory']
    if config.get('allow_delegation', False):
        backstory += "\n\nIMPORTANT: When delegating tasks, provide parameters as plain strings, not as dictionary objects."

    agents[agent_name] = Agent(
        role=config['role'],
        goal=config['goal'],
        backstory=backstory,
        tools=agent_tools,
        verbose=config.get('verbose', True),
        allow_delegation=config.get('allow_delegation', False),
        llm=llm,
        max_iter=5
    )

researcher = agents['content_researcher']
writer = agents['content_writer']
editor = agents['content_editor']

for name, agent in agents.items():
    print(f"{agent.role}: {len(agent.tools)} tool(s), delegation={'yes' if agent.allow_delegation else 'no'}")

Senior Content Researcher: 0 tool(s), delegation=no
Senior Content Writer: 0 tool(s), delegation=no
Senior Content Editor: 0 tool(s), delegation=yes


## Task Definition

Tasks are structured work items with a **description**, **expected output**, **assigned agent**, and optional **context** (dependencies on other tasks).

In [5]:
tasks_yaml_content = """
research_task:
  description: Research the latest trends and best practices in AI-powered content marketing for 2024. Focus on practical applications, case studies, and emerging technologies.
  agent: content_researcher
  expected_output: A comprehensive research report with key trends, statistics, and actionable insights.
  tools_used:
  - web_search

writing_task:
  description: Create an engaging blog post about AI-powered content marketing based on the research findings. Target audience is marketing professionals.
  agent: content_writer
  expected_output: A well-structured 1200-1500 word blog post with introduction, key sections, examples, and conclusion.
  context:
  - research_task
  tools_used: []

editing_task:
  description: Review and refine the blog post for clarity, engagement, brand consistency, and SEO optimization.
  agent: content_editor
  expected_output: A polished, publication-ready blog post with improved flow, grammar, and SEO.
  context:
  - research_task
  - writing_task
  tools_used: []
"""

tasks_config = yaml.safe_load(tasks_yaml_content.strip())

for task_name, config in tasks_config.items():
    deps = config.get('context', ['None'])
    print(f"{task_name}: agent={config['agent']}, depends_on={', '.join(deps)}")

research_task: agent=content_researcher, depends_on=None
writing_task: agent=content_writer, depends_on=research_task
editing_task: agent=content_editor, depends_on=research_task, writing_task


In [6]:
tasks = {}

for task_name, config in tasks_config.items():
    context = [tasks[dep] for dep in config.get('context', []) if dep in tasks]

    tasks[task_name] = Task(
        description=config['description'],
        agent=agents[config['agent']],
        expected_output=config['expected_output'],
        context=context if context else None
    )

research_task = tasks['research_task']
writing_task = tasks['writing_task']
editing_task = tasks['editing_task']

for i, (name, task) in enumerate(tasks.items(), 1):
    deps = [n for n, t in tasks.items() if task.context and t in (task.context if isinstance(task.context, list) else [task.context])]
    print(f"{i}. {name} -> {task.agent.role} (depends: {', '.join(deps) or 'none'})")

1. research_task -> Senior Content Researcher (depends: none)
2. writing_task -> Senior Content Writer (depends: research_task)
3. editing_task -> Senior Content Editor (depends: research_task, writing_task)


## Crew Orchestration

The **Crew** orchestrates agents and tasks. Process types: **sequential**, **hierarchical**, and **consensual**.

In [7]:
content_crew = Crew(
    agents=[researcher, writer, editor],
    tasks=[research_task, writing_task, editing_task],
    process=Process.sequential,
    verbose=True,
    memory=False,
    max_iter=10
)

print(f"Crew: {len(content_crew.agents)} agents, {len(content_crew.tasks)} tasks, process={content_crew.process.value}")

Crew: 3 agents, 3 tasks, process=sequential


## Execute Crew

In [8]:
result = content_crew.kickoff()

print("\nFinal Deliverable:")
print(result)

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 2f90ca08-4c25-4cec-be0a-3960391e7720                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Content Researcher                                                                               │
│                                                                                                                 │
│  Task: Research the latest trends and best practices in AI-powered content marketing for 2024. Focus on         │
│  practical applications, case studies, and emerging technologies.                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Content Researcher                                                                               │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Comprehensive Research Report on AI-Powered Content Marketing for 2024**                                     │
│                                                                                                                 │
│  **I. Introduction**                                                                                            │
│                                                                                                                 │
│  In 2024, AI-powered content marketing is set to revolutionize the way brands interact with their audiences.    │
│  With advancements in artificial intelligence, marketers can now create more personalized, efficient, and       │
│  effective content strategies. This report explores the latest trends, best practices, practical applications,  │
│  and emerging technologies in AI-powered content marketing.                                                     │
│                                                                                                                 │
│  **II. Key Trends**                                                                                             │
│                                                                                                                 │
│  1. **Hyper-Personalization**:                                                                                  │
│     - AI enables marketers to create highly personalized content by analyzing consumer behaviors, preferences,  │
│  and interactions in real-time. This trend involves using AI algorithms to tailor content to individual users,  │
│  enhancing engagement and conversion rates.                                                                     │
│     - **Statistics**: According to a recent survey, 72% of consumers now expect companies to anticipate their   │
│  needs and make relevant suggestions before they initiate contact.                                              │
│                                                                                                                 │
│  2. **AI-Generated Content**:                                                                                   │
│     - AI tools such as GPT-4 and Jasper are increasingly being used to generate content at scale. These tools   │
│  can write articles, social media posts, and even video scripts, allowing marketers to focus on strategy and    │
│  creativity.                                                                                                    │
│     - **Case Study**: A global fashion brand used AI-generated content to produce personalized email            │
│  campaigns, resulting in a 30% increase in open rates.                                                          │
│                                                                                                                 │
│  3. **Voice Search Optimization**:                                                                              │
│     - With the growth of voice-activated devices, optimizing content for voice search is crucial. AI helps in   │
│  understanding natural language processing (NLP) and semantic search, adapting content to meet voice search     │
│  queries.                                                                                                       │
│     - **Actionable Insight**: Incorporate conversationa

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 190bafda-7ecc-4d18-9e23-b9dff7e062fe                                                                     │
│  Agent: Senior Content Researcher                                                                               │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

Output()

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Content Editor                                                                                   │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ---                                                                                                            │
│                                                                                                                 │
│  **Embracing the Future: AI-Powered Content Marketing Trends and Best Practices for 2024**                      │
│                                                                                                                 │
│  **Introduction:**                                                                                              │
│  As we step into 2024, the landscape of content marketing continues to evolve at an unprecedented pace, driven  │
│  by the transformative power of artificial intelligence (AI). In this dynamic environment, staying ahead of     │
│  the curve is essential for marketing professionals seeking to enhance engagement and drive results. This blog  │
│  post delves into the key AI-powered content marketing trends and best practices that are set to redefine the   │
│  industry in 2024. From hyper-personalization to immersive content technologies, discover how AI can be         │
│  harnessed to create impactful marketing strategies.                                                            │
│                                                                                                                 │
│  **1. Hyper-Personalization:**                                                                                  │
│  In 2024, hyper-personalization is more than a buzzword; it’s a necessity. AI enables marketers to analyze      │
│  vast amounts of data to deliver tailored content experiences that resonate with individual users. By           │
│  leveraging machine learning algorithms, brands can predict consumer preferences and behaviors, ensuring that   │
│  each interaction is relevant and engaging. For marketing professionals, embracing hyper-personalization means  │
│  crafting content that speaks directly to the audience’s needs, fostering deeper connections and driving        │
│  conversions.                                                                                                   │
│                                                                                                                 │
│  **2. AI-Generated Content:**                                                                                   │
│  The rise of AI-generated content is revolutionizing the way marketers create and distribute content. Advanced  │
│  AI tools can produce high-quality written, visual, and audio content at scale, freeing up valuable time for    │
│  strategists to focus on creativity and innovation. In 2024, the key to success lies in integrating             │
│  AI-generated content with human oversight to maintain authenticity and brand integrity. Marketing              │
│  professionals should embrace these tools to enhance productivity and explore new content formats.              │
│                                                                                                                 │
│  **3. Voice Search Optimization:**                                                                              │
│  With the increasing popularity of voice-activated devices, optimizing content for voice search is becoming     │
│  crucial. AI-powered voice search technology is reshapi

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: f1a737da-5aa8-458c-8c4c-2be6c0a48278                                                                     │
│  Agent: Senior Content Editor                                                                                   │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 2f90ca08-4c25-4cec-be0a-3960391e7720                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: ---                                                                                              │
│                                                                                                                 │
│  **Embracing the Future: AI-Powered Content Marketing Trends and Best Practices for 2024**                      │
│                                                                                                                 │
│  **Introduction:**                                                                                              │
│  As we step into 2024, the landscape of content marketing continues to evolve at an unprecedented pace, driven  │
│  by the transformative power of artificial intelligence (AI). In this dynamic environment, staying ahead of     │
│  the curve is essential for marketing professionals seeking to enhance engagement and drive results. This blog  │
│  post delves into the key AI-powered content marketing trends and best practices that are set to redefine the   │
│  industry in 2024. From hyper-personalization to immersive content technologies, discover how AI can be         │
│  harnessed to create impactful marketing strategies.                                                            │
│                                                                                                                 │
│  **1. Hyper-Personalization:**                                                                                  │
│  In 2024, hyper-personalization is more than a buzzword; it’s a necessity. AI enables marketers to analyze      │
│  vast amounts of data to deliver tailored content experiences that resonate with individual users. By           │
│  leveraging machine learning algorithms, brands can predict consumer preferences and behaviors, ensuring that   │
│  each interaction is relevant and engaging. For marketing professionals, embracing hyper-personalization means  │
│  crafting content that speaks directly to the audience’s needs, fostering deeper connections and driving        │
│  conversions.                                                                                                   │
│                                                                                                                 │
│  **2. AI-Generated Content:**                                                                                   │
│  The rise of AI-generated content is revolutionizing the way marketers create and distribute content. Advanced  │
│  AI tools can produce high-quality written, visual, and audio content at scale, freeing up valuable time for    │
│  strategists to focus on creativity and innovation. In 2024, the key to success lies in integrating             │
│  AI-generated content with human oversight to maintain authenticity and brand integrity. Marketing              │
│  professionals should embrace these tools to enhance productivity and explore new content formats.              │
│                                                                                                                 │
│  **3. Voice Search Optimization:**                                                                              │
│  With the increasing popularity of voice-activated dev


Final Deliverable:
---

**Embracing the Future: AI-Powered Content Marketing Trends and Best Practices for 2024**

**Introduction:**  
As we step into 2024, the landscape of content marketing continues to evolve at an unprecedented pace, driven by the transformative power of artificial intelligence (AI). In this dynamic environment, staying ahead of the curve is essential for marketing professionals seeking to enhance engagement and drive results. This blog post delves into the key AI-powered content marketing trends and best practices that are set to redefine the industry in 2024. From hyper-personalization to immersive content technologies, discover how AI can be harnessed to create impactful marketing strategies.

**1. Hyper-Personalization:**  
In 2024, hyper-personalization is more than a buzzword; it’s a necessity. AI enables marketers to analyze vast amounts of data to deliver tailored content experiences that resonate with individual users. By leveraging machine learning algor

## Summary

CrewAI provides role-based multi-agent orchestration with:
- **Agents**: Role-based team members with goals, backstories, and tools
- **Tasks**: Structured work items with dependencies and expected outputs
- **Crew**: Orchestrates agents through sequential, hierarchical, or consensual processes
- **YAML config**: Scalable agent/task definitions without code changes

### Resources
- [CrewAI Documentation](https://docs.crewai.com/)
- [GitHub: crewAI-Inc/crewAI](https://github.com/crewAI-Inc/crewAI)